In [1]:
!pip install xformers bitsandbytes transformers accelerate wandb dadaptation prodigyopt -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 30.5 MB/s eta 0:

In [2]:
!pip install peft -q

In [3]:
!pip install git+https://github.com/huggingface/diffusers.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
!wget https://raw.githubusercontent.com/huggingface/diffusers/main/examples/advanced_diffusion_training/train_dreambooth_lora_sdxl_advanced.py

--2026-03-07 20:12:58--  https://raw.githubusercontent.com/huggingface/diffusers/main/examples/advanced_diffusion_training/train_dreambooth_lora_sdxl_advanced.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 109533 (107K) [text/plain]
Saving to: ‘train_dreambooth_lora_sdxl_advanced.py’

train_dreambooth_lo 100%[===================>] 106.97K  --.-KB/s    in 0.03s   

2026-03-07 20:12:58 (3.92 MB/s) - ‘train_dreambooth_lora_sdxl_advanced.py’ saved [109533/109533]



In [5]:
!pip install torchvision --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 60.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.6 requires torch<2.10,>=1.10, but you have torch 2.10.0 which is incompatible.


In [6]:
!accelerate config default

accelerate configuration saved at /root/.cache/huggingface/accelerate/default_config.yaml


In [7]:
# No need to install datasets library - we're using instance_data_dir with .txt caption files

# Your dataset is already at: /kaggle/input/neva-images/
# Format: neva_01.png + neva_01.txt, neva_02.png + neva_02.txt, etc.
# The training script automatically reads .txt files as captions when they share 
# the same name as the image.

INSTANCE_DIR = "/kaggle/input/datasets/bimukti/neva-images"  # adjust this to your actual Kaggle dataset path
OUTPUT_DIR = "/kaggle/working/neva_lora_output"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# verify dataset
images = sorted([f for f in os.listdir(INSTANCE_DIR) if f.endswith('.png')])
captions = sorted([f for f in os.listdir(INSTANCE_DIR) if f.endswith('.txt')])
print(f"Found {len(images)} images and {len(captions)} caption files")
print(f"First few: {images[:3]}")

# quick sanity check - print first caption
with open(os.path.join(INSTANCE_DIR, captions[0])) as f:
    print(f"\nSample caption ({captions[0]}):\n{f.read()}")

Found 22 images and 22 caption files
First few: ['neva_01.png', 'neva_02.png', 'neva_03.png']

Sample caption (neva_01.txt):
a TOK style panoramic mountain vista, a large bird soaring mid-flight with dark wings against sky, soft lavender and muted green palette, layered watercolor mountains with textured forest canopy below, serene and expansive


In [8]:
!pip install python-slugify
from slugify import slugify

model_name = "Neva Painterly Game Art SDXL LoRA"  # your project name
output_dir = slugify(model_name)
print(f"Output directory: {output_dir}")

Output directory: neva-painterly-game-art-sdxl-lora


In [9]:
instance_prompt = "a TOK style painting"
validation_prompt = "a TOK style side-scrolling forest environment, a young woman and white wolf running through autumn trees, warm amber palette"

In [10]:
rank = 8 # @param

In [11]:
import json, os

data_dir = "/kaggle/input/datasets/bimukti/neva-images"
work_dir = "/kaggle/working/neva_dataset"
os.makedirs(work_dir, exist_ok=True)

# copy images and build metadata
metadata = []
for f in sorted(os.listdir(data_dir)):
    if f.endswith('.png'):
        txt_file = f.replace('.png', '.txt')
        txt_path = os.path.join(data_dir, txt_file)
        
        # read caption
        if os.path.exists(txt_path):
            with open(txt_path) as t:
                caption = t.read().strip()
        else:
            caption = "a TOK style painting"
        
        # symlink image to working dir
        src = os.path.join(data_dir, f)
        dst = os.path.join(work_dir, f)
        if not os.path.exists(dst):
            os.symlink(src, dst)
        
        metadata.append({"file_name": f, "prompt": caption})

# write metadata.jsonl
with open(os.path.join(work_dir, "metadata.jsonl"), "w") as mf:
    for entry in metadata:
        mf.write(json.dumps(entry) + "\n")

print(f"Created dataset with {len(metadata)} images")
print(f"Location: {work_dir}")
print(f"\nFirst caption: {metadata[0]['prompt']}")

Created dataset with 22 images
Location: /kaggle/working/neva_dataset

First caption: a TOK style panoramic mountain vista, a large bird soaring mid-flight with dark wings against sky, soft lavender and muted green palette, layered watercolor mountains with textured forest canopy below, serene and expansive


In [12]:
!accelerate launch train_dreambooth_lora_sdxl_advanced.py \
  --pretrained_model_name_or_path="/kaggle/input/models/stabilityai/stable-diffusion-xl/pytorch/base-1-0/1" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --dataset_name="/kaggle/working/neva_dataset" \
  --caption_column="prompt" \
  --instance_prompt="a TOK style painting" \
  --validation_prompt="a TOK style side-scrolling forest environment, a young woman and white wolf running through autumn trees, warm amber palette" \
  --output_dir="$output_dir" \
  --mixed_precision="fp16" \
  --resolution=1024 \
  --train_batch_size=1 \
  --repeats=5 \
  --gradient_accumulation_steps=4 \
  --gradient_checkpointing \
  --learning_rate=1.0 \
  --text_encoder_lr=1.0 \
  --adam_beta1=0.9 \
  --adam_beta2=0.99 \
  --adam_weight_decay=0.01 \
  --optimizer="prodigy" \
  --train_text_encoder_ti \
  --train_text_encoder_ti_frac=0.5 \
  --snr_gamma=5.0 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --rank="$rank" \
  --max_train_steps=2000 \
  --checkpointing_steps=500 \
  --validation_epochs=5 \
  --seed=42 \
  --enable_xformers_memory_efficient_attention

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
You are using a model of type `clip_text_model` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
You are using a model of typ